# 고양이·오리·염소 감지 및 추적 (YOLO11 + ByteTrack)

**준비물 (구글 드라이브 `MyDrive/yolo_assignment/` 폴더에 올리기)**
- `cats-ducks-goats-v1.zip`: Roboflow에서 받은 검수 완료 데이터
- 데모용 영상 3개 (학습에 쓰지 않은 검증 영상): `KakaoTalk_261113_010117.mp4`(고양이), `KakaoTalk_261113_010013.mp4`(오리), `KakaoTalk_261113_010056.mp4`(염소)

**실행 전:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택한 뒤, 위에서부터 차례로 실행하세요.
결과(가중치, 학습 그래프, 데모 영상)는 드라이브의 `yolo_assignment/` 폴더에 저장돼서 런타임이 끊겨도 남아요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/yolo_assignment'   # 파일을 올린 드라이브 폴더
ZIP  = f'{WORK}/cats-ducks-goats-v1.zip'
import os; print(os.listdir(WORK))

In [ ]:
!pip -q install ultralytics lap

## 1. 데이터 준비: 영상 단위로 train / valid 분할
같은 영상의 프레임이 train과 valid에 섞이지 않도록 영상 ID로 나눠요. valid는 `cat_k0117`, `duck_k0013`, `goat_k0056` 세 영상이에요.

In [ ]:
import zipfile, shutil
from pathlib import Path

src = Path('/content/rf_export'); shutil.rmtree(src, ignore_errors=True)
zipfile.ZipFile(ZIP).extractall(src)

VALID_VIDEOS = {'cat_k0117', 'duck_k0013', 'goat_k0056'}
NAMES = ['cat', 'duck', 'goat']
dst = Path('/content/dataset'); shutil.rmtree(dst, ignore_errors=True)
for s in ('train', 'valid'):
    (dst / s / 'images').mkdir(parents=True); (dst / s / 'labels').mkdir(parents=True)

count = {'train': 0, 'valid': 0}
for img in src.glob('*/images/*'):
    video = '_'.join(img.name.split('_')[:2])          # cat_k0111_0743_jpg.rf.xxx.jpg -> cat_k0111
    split = 'valid' if video in VALID_VIDEOS else 'train'
    lab = img.parent.parent / 'labels' / (img.stem + '.txt')
    shutil.copy(img, dst / split / 'images' / img.name)
    if lab.exists(): shutil.copy(lab, dst / split / 'labels' / lab.name)
    count[split] += 1

(dst / 'data.yaml').write_text(f'path: {dst}\ntrain: train/images\nval: valid/images\nnames:\n' +
                               ''.join(f'  {i}: {n}\n' for i, n in enumerate(NAMES)))
print(count)   # {'train': 478, 'valid': 118} 이면 정상

## 2. 학습
T4 GPU 기준 약 15~30분 걸려요. 30 epoch 동안 성능이 나아지지 않으면 일찍 끝나요.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
model.train(data='/content/dataset/data.yaml', epochs=100, patience=30, imgsz=640, batch=16,
            device=0, project=f'{WORK}/runs', name='cdg', exist_ok=True)

## 3. 성능 확인
보고서의 '학습 과정 및 성능'에 쓸 숫자와 그래프예요.

In [ ]:
best = f'{WORK}/runs/cdg/weights/best.pt'
metrics = YOLO(best).val(data='/content/dataset/data.yaml', split='val')
print(f'mAP50 = {metrics.box.map50:.3f},  mAP50-95 = {metrics.box.map:.3f}')
for i, n in enumerate(NAMES):
    print(f'{n:5s}  AP50 = {metrics.box.ap50[i]:.3f}')

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = f'{WORK}/runs/cdg/{f}'
    if os.path.exists(p): print(f); display(Image(p, width=900))

## 4. 5초 추적 데모 영상
세 영상에서 약 1.7초씩 잘라 ByteTrack으로 추적하고 이어 붙여요. 소리는 빼고 저장해요.
`(영상, 시작초, 길이초)`의 시작초를 바꿔서 동물이 잘 움직이는 구간을 고르세요.

In [ ]:
# ---- 5초 추적 데모 영상 만들기 (Colab 노트북 셀과 동일한 코드) ----
import cv2, subprocess, os, sys
from ultralytics import YOLO

def make_demo(weights, clips, out_path, size=(720, 1280), fps=30):
    """clips: [(영상경로, 시작초, 길이초), ...] 를 차례로 추적해 하나의 영상으로 이어 붙임"""
    tmp = out_path.replace('.mp4', '_raw.mp4')
    vw = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'mp4v'), fps, size)
    for video, start, dur in clips:
        model = YOLO(weights)                       # 클립마다 새로 불러서 추적 ID 초기화
        cap = cv2.VideoCapture(video); vfps = cap.get(cv2.CAP_PROP_FPS) or 30; cap.release()
        first = int(start * vfps); n_out = round(dur * fps); k = 0
        for i, r in enumerate(model.track(source=video, stream=True, persist=True,
                                          tracker='bytetrack.yaml', conf=0.3, verbose=False)):
            if i < first: continue
            if k >= n_out: break
            frame = None
            while k < n_out and i >= first + int(k * vfps / fps):   # 영상 fps 가 달라도 길이를 정확히 맞춤
                if frame is None:
                    frame = r.plot(line_width=3)
                    h, w = frame.shape[:2]; s = min(size[0] / w, size[1] / h)
                    frame = cv2.resize(frame, (int(w * s), int(h * s)))
                    frame = cv2.copyMakeBorder(frame, 0, size[1] - frame.shape[0], 0, size[0] - frame.shape[1], cv2.BORDER_CONSTANT)
                vw.write(frame); k += 1
    vw.release()
    # 소리 없이(-an) 어디서나 재생되는 H.264 로 변환
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', tmp, '-c:v', 'libx264',
                    '-pix_fmt', 'yuv420p', '-an', out_path], check=True)
    os.remove(tmp)
    print('saved', out_path)

In [ ]:
best = f'{WORK}/runs/cdg/weights/best.pt'   # 학습된 가중치 경로 (셀을 단독 실행해도 되도록)
demos = {
    'cat':  (f'{WORK}/KakaoTalk_261113_010117.mp4', 2),   # 고양이
    'duck': (f'{WORK}/KakaoTalk_261113_010013.mp4', 8),   # 오리
    'goat': (f'{WORK}/KakaoTalk_261113_010056.mp4', 3),   # 염소
}
for name, (video, start) in demos.items():
    make_demo(best, [(video, start, 5.0)], f'{WORK}/demo_{name}_5s.mp4')

In [ ]:
from IPython.display import Video
for name in ['cat', 'duck', 'goat']:
    display(Video(f'{WORK}/demo_{name}_5s.mp4', embed=True, width=360))